# Risco de Evasão Acadêmica — ML + Lógica Fuzzy

**Disciplina:** Inteligência Artificial &nbsp;·&nbsp; **Prof.** William Malvezzi &nbsp;·&nbsp; **Grupo 2**

| Integrante | Integrante |
|---|---|
| João Pedro Nunes Neto | Luis Felipe Nunes da Fonseca Figueiredo |
| Leonardo dos Santos Silva | Luiz Phillipe de Souza Santos |
| Lucas Gabriel Pereira Guerra | |

**Entrega:** 25 de Junho de 2025

---

### Estratégia — Abordagem B (Pipeline integrado)

```
[Dados brutos]  →  [Machine Learning]  →  [P(evasão = alto)]
                                                  ↓
[frequencia]    ─────────────────────→  [Sistema Fuzzy]  →  [Score 0–100 + rótulo]
[acessos_ava]   ─────────────────────→
```

O ML captura padrões estatísticos dos dados históricos e gera uma probabilidade.
O Fuzzy converte essa probabilidade — junto com frequência e engajamento — em
linguagem compreensível para coordenadores pedagógicos agirem.

---

### Sumário

1. [Dependências](#deps)
2. [Base de Dados](#base)
3. [Análise Exploratória](#eda)
4. [Pré-processamento](#preproc)
5. [Parte 1 — Machine Learning](#ml)
6. [Parte 2 — Sistema Fuzzy](#fuzzy)
7. [Parte 3 — Integração](#integracao)
8. [Resultados e Discussão](#resultados)
9. [Conclusão](#conclusao)

---
<a id='deps'></a>

## 1. Dependências

In [ ]:
# scikit-fuzzy não vem instalada no Colab por padrão
!pip install scikit-fuzzy --quiet

import sklearn, numpy, pandas, matplotlib, skfuzzy
for lib in [sklearn, numpy, pandas, matplotlib, skfuzzy]:
    print(f'{lib.__name__:<16} {lib.__version__}')

In [ ]:
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing   import StandardScaler
from sklearn.tree            import DecisionTreeClassifier, export_text, plot_tree
from sklearn.naive_bayes     import GaussianNB
from sklearn.metrics         import (accuracy_score, confusion_matrix,
                                      classification_report, ConfusionMatrixDisplay)
import skfuzzy as fuzz

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid', palette='muted')

SEMENTE = 42
np.random.seed(SEMENTE)

---
<a id='base'></a>

## 2. Base de Dados

Base sintética balanceada com **420 registros** (140 por classe de risco).

**Por que sintética?** Dados reais de estudantes são protegidos pela LGPD.
O enunciado permite bases simuladas desde que coerentes com o domínio.
O balanceamento é uma decisão deliberada: evita que os modelos se viesem para
a classe majoritária — problema comum em dados reais de evasão.

| Variável | Tipo | Intervalo | Descrição |
|---|---|---|---|
| `frequencia` | Contínua | 0 – 100% | Percentual de presença nas aulas |
| `nota_media` | Contínua | 0 – 10 | Média geral das notas |
| `atividades_entregues` | Contínua | 0 – 100% | Atividades entregues no prazo |
| `acessos_ava` | Inteira | 0 – 30/sem | Acessos ao AVA por semana |
| `reprovas_anteriores` | Inteira | 0 – 5 | Reprovações no histórico |
| `situacao_financeira` | Ordinal | 1 · 2 · 3 | instável · regular · estável |
| `risco_evasao` | Alvo | baixo · medio · alto | Variável a predizer |

In [ ]:
N_POR_CLASSE = 140


def gerar_perfil(n, freq_mu, nota_mu, atv_mu, eng_mu,
                 pesos_reprovas, pesos_financeiro, seed_offset=0):
    """
    Gera n estudantes simulados com o perfil de risco especificado.

    Cada perfil representa um cluster real de comportamento acadêmico:
    baixo risco → alta frequência e engajamento; alto risco → o inverso.
    Os pesos de reprovas_anteriores e situacao_financeira refletem
    a distribuição observada na literatura de evasão (INEP, 2022).
    """
    np.random.seed(SEMENTE + seed_offset)
    return {
        'frequencia'          : np.clip(np.random.normal(freq_mu, 12, n), 0, 100),
        'nota_media'          : np.clip(np.random.normal(nota_mu, 1.5, n), 0, 10),
        'atividades_entregues': np.clip(np.random.normal(atv_mu,  15, n), 0, 100),
        'acessos_ava'         : np.clip(np.random.poisson(eng_mu,     n), 0, 30).astype(int),
        'reprovas_anteriores' : np.random.choice([0, 1, 2, 3, 4, 5], n, p=pesos_reprovas),
        'situacao_financeira' : np.random.choice([1, 2, 3],           n, p=pesos_financeiro),
    }


# Três perfis distintos — separabilidade realista entre as classes
PERFIS = [
    (gerar_perfil(N_POR_CLASSE,
                  freq_mu=88, nota_mu=7.8, atv_mu=87, eng_mu=14,
                  pesos_reprovas=[.70, .20, .07, .02, .01, .00],
                  pesos_financeiro=[.10, .35, .55],
                  seed_offset=0), 'baixo'),

    (gerar_perfil(N_POR_CLASSE,
                  freq_mu=70, nota_mu=5.8, atv_mu=65, eng_mu=8,
                  pesos_reprovas=[.35, .30, .20, .10, .04, .01],
                  pesos_financeiro=[.25, .45, .30],
                  seed_offset=1), 'medio'),

    (gerar_perfil(N_POR_CLASSE,
                  freq_mu=48, nota_mu=3.8, atv_mu=40, eng_mu=4,
                  pesos_reprovas=[.10, .20, .25, .25, .12, .08],
                  pesos_financeiro=[.45, .35, .20],
                  seed_offset=2), 'alto'),
]

dfs = []
for dados, classe in PERFIS:
    df_parcial = pd.DataFrame(dados)
    df_parcial['risco_evasao'] = classe
    dfs.append(df_parcial)

df = (pd.concat(dfs, ignore_index=True)
        .sample(frac=1, random_state=SEMENTE)
        .reset_index(drop=True))

df[['frequencia', 'nota_media', 'atividades_entregues']] = (
    df[['frequencia', 'nota_media', 'atividades_entregues']].round(1)
)

df.to_csv('base_evasao_academica.csv', index=False)

print(f'Registros: {len(df)} | Classes: {df.risco_evasao.value_counts().to_dict()}')
df.head()

---
<a id='eda'></a>

## 3. Análise Exploratória dos Dados

A EDA revela distribuições, anomalias e o poder discriminativo de cada variável.
Essas informações guiam as decisões de pré-processamento e a escolha dos algoritmos.

In [ ]:
# ─── Estatísticas descritivas e valores faltantes ─────────────────────────────

display(df.describe().round(2))

ausentes = df.isnull().sum()
print(f'\nValores faltantes: {ausentes.sum()}')
print('→ Base sintética: nenhum.')
print('→ Em dados reais: mediana para numéricas, moda para categóricas.')

In [ ]:
# ─── Distribuição das variáveis numéricas ─────────────────────────────────────

COLUNAS_NUMERICAS = ['frequencia', 'nota_media', 'atividades_entregues',
                     'acessos_ava', 'reprovas_anteriores']
PALETA_HIST       = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2']
PALETA_RISCO      = {'baixo': '#55A868', 'medio': '#DD8452', 'alto': '#C44E52'}

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, (coluna, cor) in enumerate(zip(COLUNAS_NUMERICAS, PALETA_HIST)):
    axes[i].hist(df[coluna], bins=26, color=cor, alpha=0.75, edgecolor='white')
    axes[i].axvline(df[coluna].mean(),   color='black', lw=1.8, ls='--',
                    label=f'média={df[coluna].mean():.1f}')
    axes[i].axvline(df[coluna].median(), color='red',   lw=1.4, ls=':',
                    label=f'mediana={df[coluna].median():.1f}')
    axes[i].set_title(coluna.replace('_', ' ').title(), fontweight='bold')
    axes[i].legend(fontsize=8)

# Variável-alvo no sexto painel
contagem = df.risco_evasao.value_counts()
barras   = axes[5].bar(contagem.index, contagem.values,
                       color=[PALETA_RISCO[c] for c in contagem.index],
                       edgecolor='white')
for barra, valor in zip(barras, contagem.values):
    axes[5].text(barra.get_x() + barra.get_width() / 2, valor + 1,
                 str(valor), ha='center', fontweight='bold')
axes[5].set_title('Variável-Alvo: risco_evasao', fontweight='bold')

fig.suptitle('Distribuição das Variáveis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ─── Boxplots por classe de risco ─────────────────────────────────────────────
# Boxes bem separadas = boa capacidade discriminativa da variável.
# O losango branco marca a média — complementa a mediana do box.

ORDEM_CLASSES = ['baixo', 'medio', 'alto']

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, coluna in zip(axes, ['frequencia', 'nota_media', 'atividades_entregues']):
    sns.boxplot(data=df, x='risco_evasao', y=coluna,
                order=ORDEM_CLASSES, palette=PALETA_RISCO, ax=ax)
    medias = df.groupby('risco_evasao')[coluna].mean()
    for j, classe in enumerate(ORDEM_CLASSES):
        ax.scatter(j, medias[classe],
                   marker='D', color='white', edgecolor='black', s=35, zorder=5)
    ax.set_title(f'{coluna.replace("_", " ").title()} × Risco', fontweight='bold')

fig.suptitle('Separabilidade por Classe de Risco', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('frequencia e nota_media mostram separação clara entre as três classes.')

In [ ]:
# ─── Correlação de Pearson ────────────────────────────────────────────────────
# |r| > 0.85 entre duas features indicaria multicolinearidade grave.

fig, ax = plt.subplots(figsize=(7, 5))
correlacao = df[COLUNAS_NUMERICAS].corr()
sns.heatmap(correlacao,
            annot=True, fmt='.2f', cmap='coolwarm',
            mask=np.triu(np.ones_like(correlacao, dtype=bool)),
            ax=ax, linewidths=0.5, vmin=-1, vmax=1)
ax.set_title('Correlação de Pearson', fontweight='bold')
plt.tight_layout()
plt.show()

print('Nenhum par com |r| > 0.85 → sem multicolinearidade grave.')
print('Todas as features podem ser mantidas.')

In [ ]:
# ─── Scatter: frequência × nota_media ────────────────────────────────────────
# Os dois preditores mais fortes. Visualiza a separabilidade espacial.

fig, ax = plt.subplots(figsize=(8, 5))
for classe in ORDEM_CLASSES:
    subset = df[df.risco_evasao == classe]
    ax.scatter(subset.frequencia, subset.nota_media,
               c=PALETA_RISCO[classe], alpha=0.45, s=20,
               label=classe.capitalize())

# Limiares pedagógicos de referência
ax.axvline(60,  color='gray', ls='--', lw=1.4, alpha=0.7, label='Freq. mínima (60%)')
ax.axhline(5.0, color='gray', ls=':',  lw=1.4, alpha=0.7, label='Nota mínima (5.0)')
ax.set_xlabel('Frequência (%)')
ax.set_ylabel('Nota Média')
ax.set_title('Separabilidade: Frequência × Nota', fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

---
<a id='preproc'></a>

## 4. Pré-processamento

| Etapa | Técnica | Motivo |
|---|---|---|
| Codificação do alvo | Mapeamento manual `baixo=0, medio=1, alto=2` | Preserva a ordem semântica |
| Normalização | `StandardScaler` (z-score) | Elimina diferença de escala entre features |
| Divisão | 70% treino / 30% teste estratificado | Proporção de classes igual nos dois conjuntos |

> A Árvore de Decisão é invariante à escala e usa os dados originais.
> O Naive Bayes usa os dados normalizados para estabilidade numérica.

In [ ]:
FEATURES      = ['frequencia', 'nota_media', 'atividades_entregues',
                 'acessos_ava', 'reprovas_anteriores', 'situacao_financeira']
NOMES_CLASSES = ['baixo', 'medio', 'alto']
MAPA_CLASSE   = {'baixo': 0, 'medio': 1, 'alto': 2}
MAPA_INVERSO  = {0: 'baixo', 1: 'medio', 2: 'alto'}

X = df[FEATURES]
y = df['risco_evasao'].map(MAPA_CLASSE)

# Divisão estratificada — mantém proporção de classes em treino e teste
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.30, random_state=SEMENTE, stratify=y
)

# fit apenas no treino — evita data leakage para o conjunto de teste
normalizador = StandardScaler()
X_treino_norm = normalizador.fit_transform(X_treino)
X_teste_norm  = normalizador.transform(X_teste)

print(f'Treino : {len(X_treino)} amostras')
print(f'Teste  : {len(X_teste)} amostras')
print()
print('Distribuição no treino:', pd.Series(y_treino).map(MAPA_INVERSO).value_counts().to_dict())
print('Distribuição no teste :', pd.Series(y_teste).map(MAPA_INVERSO).value_counts().to_dict())

---
<a id='ml'></a>

## 5. Parte 1 — Machine Learning

Dois algoritmos preferidos pela disciplina para comparação interna:

| Algoritmo | Justificativa |
|---|---|
| **Árvore de Decisão** | Alta interpretabilidade; as regras IF-THEN geradas dialogam diretamente com as regras fuzzy |
| **Naive Bayes Gaussiano** | Probabilístico por natureza; `predict_proba()` fornece probabilidades calibradas para alimentar o pipeline Fuzzy |

### Métricas utilizadas

| Métrica | O que mede |
|---|---|
| **Acurácia** | Proporção de predições corretas — pode ser enganosa em bases desbalanceadas |
| **Precisão macro** | Dos preditos como classe X, quantos realmente são X — penaliza falsos positivos |
| **Recall macro** | Dos que são classe X, quantos o modelo encontrou — penaliza falsos negativos |
| **F1-score macro** | Média harmônica entre precisão e recall — métrica mais equilibrada para multiclasse |
| **CV 5-fold** | Estima a capacidade de generalização e detecta overfitting |

In [ ]:
def avaliar_modelo(nome, modelo, X_tr, y_tr, X_te, y_te):
    """
    Treina o modelo, calcula métricas completas e plota a matriz de confusão.

    Usa validação cruzada estratificada de 5 folds sobre o conjunto de treino
    para estimar a capacidade de generalização real do modelo.

    Returns:
        tuple: (modelo treinado, acurácia, relatório dict, scores CV)
    """
    modelo.fit(X_tr, y_tr)
    predicoes = modelo.predict(X_te)

    acuracia  = accuracy_score(y_te, predicoes)
    relatorio = classification_report(y_te, predicoes,
                                       target_names=NOMES_CLASSES, output_dict=True)
    scores_cv = cross_val_score(
        modelo, X_tr, y_tr,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMENTE),
        scoring='f1_macro'
    )

    print(f'─── {nome} ──────────────────────────────────────')
    print(f'Acurácia    : {acuracia:.4f}  ({acuracia * 100:.1f}%)')
    print(f'F1 macro    : {relatorio["macro avg"]["f1-score"]:.4f}')
    print(f'CV F1 5-fold: {scores_cv.mean():.4f} ± {scores_cv.std():.4f}')
    print()
    print(classification_report(y_te, predicoes, target_names=NOMES_CLASSES))

    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay(
        confusion_matrix=confusion_matrix(y_te, predicoes),
        display_labels=NOMES_CLASSES
    ).plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'Matriz de Confusão — {nome}', fontweight='bold')
    plt.tight_layout()
    plt.show()

    return modelo, acuracia, relatorio, scores_cv

In [ ]:
# ─── Árvore de Decisão ────────────────────────────────────────────────────────
#
# max_depth=6        → controla overfitting; árvores muito profundas
#                      memorizam o treino sem generalizar
# min_samples_leaf=5 → cada folha precisa de ao menos 5 exemplos;
#                      decisões baseadas em poucos dados são instáveis
# criterion='gini'   → impureza de Gini; computacionalmente mais eficiente
#                      que entropia e produz resultados equivalentes
# class_weight       → compensa qualquer desbalanceamento residual entre classes

arvore = DecisionTreeClassifier(
    max_depth=6,
    min_samples_leaf=5,
    criterion='gini',
    class_weight='balanced',
    random_state=SEMENTE
)

# Árvores são invariantes à escala — usa dados originais sem normalização
arvore_modelo, arvore_acc, arvore_rep, arvore_cv = avaliar_modelo(
    'Árvore de Decisão', arvore, X_treino, y_treino, X_teste, y_teste
)

In [ ]:
# ─── Visualização da Árvore ───────────────────────────────────────────────────
# A estrutura visual é uma das maiores vantagens desse algoritmo:
# é possível auditar exatamente como cada decisão é tomada.

fig, ax = plt.subplots(figsize=(22, 10))
plot_tree(arvore_modelo,
          feature_names=FEATURES,
          class_names=NOMES_CLASSES,
          filled=True, rounded=True,
          fontsize=7, ax=ax,
          impurity=True, proportion=False, precision=2)
ax.set_title('Árvore de Decisão — Risco de Evasão Acadêmica',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Extrato textual das primeiras regras
regras_texto = export_text(arvore_modelo, feature_names=FEATURES).split('\n')
print('\n'.join(regras_texto[:30]))
print('...')

In [ ]:
# ─── Importância das features ─────────────────────────────────────────────────
# Mede quanto cada variável contribuiu para reduzir a impureza de Gini
# ao longo de todas as divisões da árvore.

importancias = pd.Series(
    arvore_modelo.feature_importances_, index=FEATURES
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(7, 4))
cores = ['#C44E52' if v >= importancias.median() else '#4C72B0'
         for v in importancias]
barras = ax.barh(importancias.index, importancias.values,
                 color=cores, edgecolor='white')
for barra, valor in zip(barras, importancias.values):
    ax.text(valor + 0.003, barra.get_y() + barra.get_height() / 2,
            f'{valor:.3f}', va='center', fontsize=9)
ax.set_xlabel('Importância (redução de impureza Gini)')
ax.set_title('Importância das Features — Árvore de Decisão', fontweight='bold')
ax.set_xlim(0, importancias.max() * 1.18)
plt.tight_layout()
plt.show()

In [ ]:
# ─── Naive Bayes Gaussiano ────────────────────────────────────────────────────
#
# Aplica o Teorema de Bayes para estimação da probabilidade posterior:
#   P(classe | features) ∝ P(features | classe) × P(classe)
#
# O qualificativo "Naive" (ingênuo) vem da suposição de independência
# condicional entre as features dado a classe. Essa simplificação raramente
# é verdadeira, mas produz resultados surpreendentemente bons na prática.
#
# A versão Gaussiana assume distribuição normal para features contínuas —
# adequada para frequencia, nota_media e atividades_entregues.
#
# Papel no pipeline: predict_proba() retorna probabilidades calibradas para
# cada classe; a probabilidade P(alto) alimenta o sistema Fuzzy na Parte 3.

naive_bayes = GaussianNB(var_smoothing=1e-9)  # suavização evita P=0 por variância nula

# Naive Bayes usa dados normalizados
naive_bayes_modelo, nb_acc, nb_rep, nb_cv = avaliar_modelo(
    'Naive Bayes Gaussiano', naive_bayes,
    X_treino_norm, y_treino,
    X_teste_norm,  y_teste
)

In [ ]:
# ─── Comparação entre os modelos ─────────────────────────────────────────────

df_comparacao = pd.DataFrame({
    'Modelo'       : ['Árvore de Decisão', 'Naive Bayes'],
    'Acurácia'     : [arvore_acc, nb_acc],
    'F1 macro'     : [arvore_rep['macro avg']['f1-score'], nb_rep['macro avg']['f1-score']],
    'Precisão'     : [arvore_rep['macro avg']['precision'], nb_rep['macro avg']['precision']],
    'Recall'       : [arvore_rep['macro avg']['recall'], nb_rep['macro avg']['recall']],
    'CV F1 média'  : [arvore_cv.mean(), nb_cv.mean()],
    'CV F1 ±std'   : [arvore_cv.std(),  nb_cv.std()],
}).round(4)
print(df_comparacao.to_string(index=False))

metricas = ['Acurácia', 'F1 macro', 'Precisão', 'Recall']
x, largura = np.arange(len(metricas)), 0.32

fig, ax = plt.subplots(figsize=(10, 4))
for i, (nome, cor) in enumerate([('Árvore de Decisão', '#4C72B0'), ('Naive Bayes', '#DD8452')]):
    barras = ax.bar(x + largura * (i - 0.5), df_comparacao.iloc[i][metricas],
                    largura, label=nome, color=cor, alpha=0.85, edgecolor='white')
    for b in barras:
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.004,
                f'{b.get_height():.3f}', ha='center', fontsize=8.5)
ax.set_xticks(x); ax.set_xticklabels(metricas)
ax.set_ylim(0, 1.12)
ax.axhline(0.80, color='gray', ls='--', alpha=0.4, label='Ref. 80%')
ax.set_title('Comparação de Métricas — Árvore × Naive Bayes', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

# Seleciona o melhor modelo pelo F1 macro para a integração Fuzzy
melhor_nome   = df_comparacao.iloc[df_comparacao['F1 macro'].idxmax()]['Modelo']
melhor_modelo = naive_bayes_modelo if 'Naive' in melhor_nome else arvore_modelo
X_teste_melhor = X_teste_norm if 'Naive' in melhor_nome else X_teste

print(f'\nModelo selecionado para o pipeline Fuzzy: {melhor_nome}')

---
<a id='fuzzy'></a>

## 6. Parte 2 — Sistema de Inferência Fuzzy

A Lógica Fuzzy (Zadeh, 1965) permite pertinência parcial a múltiplos conjuntos.
Uma frequência de 62% pode pertencer com grau 0.7 ao conjunto *baixa* e
com grau 0.3 ao conjunto *média* simultaneamente — diferente da lógica
clássica, que seria simplesmente "acima de 60%".

### Arquitetura — Método Mamdani

```
Variáveis de entrada                Variável de saída
─────────────────────               ─────────────────────────────────
prob_evasao_ml  (0–1)   ─┐
frequencia      (0–100%) ├──▶  [8 regras SE…ENTÃO]  ──▶  risco_final (0–100)
engajamento_ava (0–30)   ─┘        Mamdani                + rótulo linguístico
```

**Processo de inferência:**
1. **Fuzzificação** — converter valores numéricos em graus de pertinência μ(x)
2. **Aplicação das regras** — ativação de cada regra com operador E = mínimo
3. **Implicação** — corte da MF consequente no nível da ativação
4. **Agregação** — máximo dos consequentes cortados
5. **Defuzzificação** — centroide da área agregada → score numérico

| Variável | Termos linguísticos | Função de pertinência |
|---|---|---|
| `prob_evasao_ml` | baixa · média · alta | Trapezoidal (bordas) + Triangular (centro) |
| `frequencia` | baixa · média · alta | Trapezoidal + Triangular |
| `engajamento_ava` | baixo · médio · alto | Trapezoidal + Triangular |
| `risco_final` | baixo · médio · alto · crítico | Trapezoidal + Triangular |

In [ ]:
# ─── Universos de discurso ────────────────────────────────────────────────────
# Cada universo define o domínio contínuo de uma variável fuzzy.
# Usamos 200 pontos — resolução suficiente para defuzzificação precisa.

universo_prob_ml    = np.linspace(0,   1, 200)  # probabilidade do modelo: 0 a 1
universo_frequencia = np.linspace(0, 100, 200)  # presença em aula: 0% a 100%
universo_engajamento= np.linspace(0,  30, 200)  # acessos ao AVA/semana: 0 a 30
universo_risco      = np.linspace(0, 100, 200)  # score de saída: 0 a 100

# ─── prob_evasao_ml ───────────────────────────────────────────────────────────
# Trapezoidais nas extremidades capturam a certeza nas bordas do universo.
# Triangular no centro modela a transição gradual entre baixa e alta.

prob_baixa = fuzz.trapmf(universo_prob_ml, [0.00, 0.00, 0.20, 0.40])
prob_media = fuzz.trimf (universo_prob_ml, [0.25, 0.50, 0.75])
prob_alta  = fuzz.trapmf(universo_prob_ml, [0.60, 0.80, 1.00, 1.00])

# ─── frequencia ───────────────────────────────────────────────────────────────
# Limiar pedagógico de 75% de frequência mínima exigida.
# Abaixo de 65% → predominantemente "baixa".

freq_baixa = fuzz.trapmf(universo_frequencia, [ 0,  0, 40, 65])
freq_media = fuzz.trimf (universo_frequencia, [50, 70, 85])
freq_alta  = fuzz.trapmf(universo_frequencia, [75, 90, 100, 100])

# ─── engajamento_ava ──────────────────────────────────────────────────────────
# Menos de 4 acessos/semana → praticamente ausente no AVA.
# Acima de 15 → perfil consistentemente engajado.

eng_baixo = fuzz.trapmf(universo_engajamento, [ 0,  0,  4,  9])
eng_medio = fuzz.trimf (universo_engajamento, [ 6, 12, 18])
eng_alto  = fuzz.trapmf(universo_engajamento, [15, 22, 30, 30])

# ─── risco_final (saída) ──────────────────────────────────────────────────────
# Quatro termos permitem maior granularidade na comunicação do risco.
# "Crítico" sinaliza urgência máxima — score acima de 75.

risco_baixo   = fuzz.trapmf(universo_risco, [ 0,  0, 20, 35])
risco_medio   = fuzz.trimf (universo_risco, [25, 45, 65])
risco_alto    = fuzz.trimf (universo_risco, [50, 68, 82])
risco_critico = fuzz.trapmf(universo_risco, [75, 88, 100, 100])

print('Funções de pertinência definidas.')

In [ ]:
# ─── Visualização das funções de pertinência ──────────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

CONFIGS_MF = [
    (axes[0, 0], universo_prob_ml,
     [(prob_baixa, 'Baixa', '#55A868', '-'),
      (prob_media, 'Média', '#4C72B0', '--'),
      (prob_alta,  'Alta',  '#C44E52', '-.')],
     'Entrada 1: Probabilidade ML (0–1)', 'Probabilidade'),

    (axes[0, 1], universo_frequencia,
     [(freq_baixa, 'Baixa', '#C44E52', '-'),
      (freq_media, 'Média', '#4C72B0', '--'),
      (freq_alta,  'Alta',  '#55A868', '-.')],
     'Entrada 2: Frequência (%)', 'Frequência (%)'),

    (axes[1, 0], universo_engajamento,
     [(eng_baixo, 'Baixo', '#C44E52', '-'),
      (eng_medio, 'Médio', '#4C72B0', '--'),
      (eng_alto,  'Alto',  '#55A868', '-.')],
     'Entrada 3: Engajamento AVA (acessos/semana)', 'Acessos por semana'),

    (axes[1, 1], universo_risco,
     [(risco_baixo,   'Baixo',   '#55A868', '-'),
      (risco_medio,   'Médio',   '#4C72B0', '--'),
      (risco_alto,    'Alto',    '#DD8452', '-.'),
      (risco_critico, 'Crítico', '#C44E52', ':')],
     'Saída: Risco Final (score 0–100)', 'Score'),
]

for ax, universo, funcoes, titulo, xlabel in CONFIGS_MF:
    for mf, rotulo, cor, estilo in funcoes:
        ax.plot(universo, mf, estilo, color=cor, lw=2.0, label=rotulo)
        ax.fill_between(universo, mf, alpha=0.08, color=cor)
    ax.set_title(titulo, fontweight='bold', fontsize=10)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('μ(x)')
    ax.set_ylim(-0.05, 1.12)
    ax.legend(fontsize=9)

fig.suptitle('Funções de Pertinência — Sistema Fuzzy', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ─── Catálogo de regras fuzzy ─────────────────────────────────────────────────

CATALOGO_REGRAS = [
    ('R1', 'prob_alta   E freq_baixa             ', 'CRÍTICO',
     'prob. alta + baixa presença → situação de emergência'),
    ('R2', 'prob_alta   E eng_baixo              ', 'CRÍTICO',
     'prob. alta + sem acesso ao AVA → aluno desapareceu'),
    ('R3', 'prob_alta   E freq_media             ', 'ALTO',
     'prob. alta com frequência mediana → risco elevado'),
    ('R4', 'prob_media  E freq_baixa             ', 'ALTO',
     'risco médio do ML agravado por baixa frequência'),
    ('R5', 'prob_media  E eng_medio              ', 'MÉDIO',
     'engajamento regular atenua o risco moderado'),
    ('R6', 'prob_media  E freq_alta  E eng_medio ', 'MÉDIO',
     'frequência alta com eng. médio → risco controlável'),
    ('R7', 'prob_baixa  E freq_alta              ', 'BAIXO',
     'ML confiante + alta presença → perfil seguro'),
    ('R8', 'prob_baixa  E eng_alto               ', 'BAIXO',
     'ML confiante + muito ativo no AVA → perfil de sucesso'),
]

print(f'{"ID":4}  {"Condição SE":44} {"Então":9}  Justificativa')
print('─' * 95)
for rid, condicao, consequente, justificativa in CATALOGO_REGRAS:
    print(f'{rid:4}  SE {condicao:44} → {consequente:9}  [{justificativa}]')

In [ ]:
def executar_inferencia(probabilidade_ml: float,
                        frequencia: float,
                        engajamento: float) -> dict:
    """
    Executa o pipeline completo de inferência Mamdani para um estudante.

    Etapas:
        1. Fuzzificação — graus de pertinência μ para cada entrada
        2. Aplicação das regras — ativação via operador E = mínimo
        3. Implicação — corte da MF consequente no nível da ativação
        4. Agregação — máximo dos consequentes cortados
        5. Defuzzificação — centroide da área agregada

    Args:
        probabilidade_ml: P(evasão = alto) gerado pelo modelo de ML (0–1)
        frequencia: percentual de presença do estudante (0–100)
        engajamento: acessos ao AVA por semana (0–30)

    Returns:
        dict com score (0–100), rótulo linguístico, área agregada e ativações
    """
    interp = fuzz.interp_membership

    # ─── 1. Fuzzificação ──────────────────────────────────────────────────────
    mu_prob_baixa = interp(universo_prob_ml,    prob_baixa, probabilidade_ml)
    mu_prob_media = interp(universo_prob_ml,    prob_media, probabilidade_ml)
    mu_prob_alta  = interp(universo_prob_ml,    prob_alta,  probabilidade_ml)

    mu_freq_baixa = interp(universo_frequencia, freq_baixa, frequencia)
    mu_freq_media = interp(universo_frequencia, freq_media, frequencia)
    mu_freq_alta  = interp(universo_frequencia, freq_alta,  frequencia)

    mu_eng_baixo  = interp(universo_engajamento, eng_baixo, engajamento)
    mu_eng_medio  = interp(universo_engajamento, eng_medio, engajamento)
    mu_eng_alto   = interp(universo_engajamento, eng_alto,  engajamento)

    # ─── 2. Aplicação das regras (operador E = mínimo) ────────────────────────
    ativacao_r1 = np.fmin(mu_prob_alta,  mu_freq_baixa)                          # → CRÍTICO
    ativacao_r2 = np.fmin(mu_prob_alta,  mu_eng_baixo)                           # → CRÍTICO
    ativacao_r3 = np.fmin(mu_prob_alta,  mu_freq_media)                          # → ALTO
    ativacao_r4 = np.fmin(mu_prob_media, mu_freq_baixa)                          # → ALTO
    ativacao_r5 = np.fmin(mu_prob_media, mu_eng_medio)                           # → MÉDIO
    ativacao_r6 = np.fmin(np.fmin(mu_prob_media, mu_freq_alta), mu_eng_medio)   # → MÉDIO
    ativacao_r7 = np.fmin(mu_prob_baixa, mu_freq_alta)                           # → BAIXO
    ativacao_r8 = np.fmin(mu_prob_baixa, mu_eng_alto)                            # → BAIXO

    # ─── 3. Implicação — corta cada MF consequente no nível de ativação ───────
    impl_critico_r1 = np.fmin(ativacao_r1, risco_critico)
    impl_critico_r2 = np.fmin(ativacao_r2, risco_critico)
    impl_alto_r3    = np.fmin(ativacao_r3, risco_alto)
    impl_alto_r4    = np.fmin(ativacao_r4, risco_alto)
    impl_medio_r5   = np.fmin(ativacao_r5, risco_medio)
    impl_medio_r6   = np.fmin(ativacao_r6, risco_medio)
    impl_baixo_r7   = np.fmin(ativacao_r7, risco_baixo)
    impl_baixo_r8   = np.fmin(ativacao_r8, risco_baixo)

    # ─── 4. Agregação — máximo entre regras que ativam o mesmo consequente ────
    agg_critico = np.fmax(impl_critico_r1, impl_critico_r2)
    agg_alto    = np.fmax(impl_alto_r3,    impl_alto_r4)
    agg_medio   = np.fmax(impl_medio_r5,   impl_medio_r6)
    agg_baixo   = np.fmax(impl_baixo_r7,   impl_baixo_r8)

    area_agregada = np.fmax(np.fmax(agg_critico, agg_alto),
                             np.fmax(agg_medio,   agg_baixo))

    # ─── 5. Defuzzificação — centroide da área agregada ───────────────────────
    # ∫(x · μ(x) dx) / ∫(μ(x) dx)
    if area_agregada.sum() == 0:
        score = 50.0  # fallback: risco indeterminado — sem regras ativadas
    else:
        score = fuzz.defuzz(universo_risco, area_agregada, 'centroid')

    # Rótulo linguístico baseado no score
    if   score < 30: rotulo = 'baixo'
    elif score < 55: rotulo = 'medio'
    elif score < 75: rotulo = 'alto'
    else:            rotulo = 'critico'

    return {
        'score'         : round(score, 2),
        'rotulo'        : rotulo,
        'area_agregada' : area_agregada,
        'ativacoes'     : {
            'R1_critico': round(ativacao_r1, 4),
            'R2_critico': round(ativacao_r2, 4),
            'R3_alto'   : round(ativacao_r3, 4),
            'R4_alto'   : round(ativacao_r4, 4),
            'R5_medio'  : round(ativacao_r5, 4),
            'R6_medio'  : round(ativacao_r6, 4),
            'R7_baixo'  : round(ativacao_r7, 4),
            'R8_baixo'  : round(ativacao_r8, 4),
        },
    }

print('executar_inferencia() definida (8 regras, Mamdani, centroide).')

In [ ]:
# ─── Casos de teste manuais ───────────────────────────────────────────────────
# Valida a coerência do sistema antes de aplicar ao conjunto de teste completo.
# Testamos perfis extremos e intermediários representativos.

CASOS_TESTE = [
    (0.90, 28,  2, 'critico',    'CRÍTICO — ML alto + freq baixa + eng mínimo'),
    (0.75, 52,  5, 'critico',    'ALTO    — ML alto + freq média + eng baixo'),
    (0.45, 68, 11, 'medio',      'MÉDIO   — ML médio + freq média + eng médio'),
    (0.18, 91, 22, 'baixo',      'BAIXO   — ML baixo + freq alta + eng alto'),
    (0.55, 73,  9, 'medio|alto', 'LIMIAR  — valores nas zonas de transição'),
]

print(f'{"Perfil":<52}  {"Score":>6}  {"Rótulo":>8}  OK?')
print('─' * 80)
for prob, freq, eng, esperado, descricao in CASOS_TESTE:
    resultado = executar_inferencia(prob, freq, eng)
    ok = '✅' if resultado['rotulo'] in esperado else '⚠️ '
    print(f'{descricao:<52}  {resultado["score"]:>6.1f}  '
          f'{resultado["rotulo"].upper():>8}  {ok}')

print()
print('Nota: perfis limítrofes produzem transições graduais — comportamento correto do Fuzzy.')

In [ ]:
# ─── Superfície de controle ───────────────────────────────────────────────────
# Mostra como o score varia em todo o espaço prob_ml × frequência.
# Engajamento fixado em 10 (valor médio) para viabilizar a visualização 3D.

ENG_FIXADO = 10
grades_prob = np.linspace(0,   1, 35)
grades_freq = np.linspace(0, 100, 35)
P, F = np.meshgrid(grades_prob, grades_freq)
Z    = np.array([[executar_inferencia(P[i, j], F[i, j], ENG_FIXADO)['score']
                  for j in range(35)]
                 for i in range(35)])

fig = plt.figure(figsize=(13, 5))

ax3d = fig.add_subplot(121, projection='3d')
ax3d.plot_surface(P, F, Z, cmap='RdYlGn_r', alpha=0.88, linewidth=0)
ax3d.set_xlabel('Prob. ML');       ax3d.set_ylabel('Frequência (%)')
ax3d.set_zlabel('Score Fuzzy')
ax3d.set_title(f'Superfície de Controle\n(engajamento fixado = {ENG_FIXADO})',
               fontweight='bold')

ax2d = fig.add_subplot(122)
imagem = ax2d.contourf(P, F, Z, levels=22, cmap='RdYlGn_r')
fig.colorbar(imagem, ax=ax2d, label='Score (0–100)')
# Isocurvas nos limiares de decisão
iso = ax2d.contour(P, F, Z, levels=[30, 55, 75],
                   colors='black', linewidths=1.2, linestyles=['--', '-.', ':'])
ax2d.clabel(iso, fmt='%.0f', fontsize=8)
ax2d.set_xlabel('Prob. ML'); ax2d.set_ylabel('Frequência (%)')
ax2d.set_title('Mapa de Calor (limiares: 30 / 55 / 75)', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ─── Demonstração do processo de defuzzificação ───────────────────────────────
# Passo a passo visual para um perfil de alto risco.

ENTRADA_DEMO = (0.78, 42, 5)   # (prob_ml, frequencia, engajamento)
resultado_demo = executar_inferencia(*ENTRADA_DEMO)

fig, (painel_defuzz, painel_ativacoes) = plt.subplots(1, 2, figsize=(13, 5))

# Painel esquerdo: área agregada + centroide
for mf, rotulo, cor, estilo in [
    (risco_baixo,   'Baixo',   '#55A868', '-'),
    (risco_medio,   'Médio',   '#4C72B0', '--'),
    (risco_alto,    'Alto',    '#DD8452', '-.'),
    (risco_critico, 'Crítico', '#C44E52', ':'),
]:
    painel_defuzz.plot(universo_risco, mf, estilo, color=cor, lw=1.8, alpha=0.6, label=rotulo)

painel_defuzz.fill_between(universo_risco, resultado_demo['area_agregada'],
                            alpha=0.38, color='purple', label='Área Agregada')
painel_defuzz.axvline(resultado_demo['score'], color='black', lw=2.2, ls='--',
                      label=f'Centroide = {resultado_demo["score"]:.1f}')
painel_defuzz.set_title(
    f'Defuzzificação → {resultado_demo["rotulo"].upper()} ({resultado_demo["score"]:.1f}/100)',
    fontweight='bold')
painel_defuzz.set_xlabel('Score de Risco')
painel_defuzz.set_ylabel('μ(x)')
painel_defuzz.legend(fontsize=8)

# Painel direito: ativação de cada regra
nomes_regras   = list(resultado_demo['ativacoes'].keys())
valores_regras = list(resultado_demo['ativacoes'].values())
cores_regras   = ['#C44E52' if 'critico' in n else '#DD8452' if 'alto' in n
                  else '#4C72B0' if 'medio' in n else '#55A868'
                  for n in nomes_regras]
painel_ativacoes.barh(nomes_regras, valores_regras, color=cores_regras, edgecolor='white')
for j, valor in enumerate(valores_regras):
    painel_ativacoes.text(valor + 0.01, j, f'{valor:.4f}', va='center', fontsize=9)
painel_ativacoes.set_xlim(0, 1.2)
painel_ativacoes.set_title('Ativação das Regras', fontweight='bold')

descricao_entrada = (f'prob_ML={ENTRADA_DEMO[0]} | '
                     f'freq={ENTRADA_DEMO[1]}% | eng={ENTRADA_DEMO[2]}')
fig.suptitle(f'Passo a Passo da Inferência Fuzzy\n{descricao_entrada}',
             fontweight='bold')
plt.tight_layout()
plt.show()

# Saída textual do processo
print(f'Entrada : {descricao_entrada}')
print(f'Score   : {resultado_demo["score"]:.2f} / 100')
print(f'Rótulo  : {resultado_demo["rotulo"].upper()}')
print()
print('Ativações por regra:')
for regra, ativacao in resultado_demo['ativacoes'].items():
    barra = '█' * int(ativacao * 25)
    print(f'  {regra:15}: {ativacao:.4f}  {barra}')

---
<a id='integracao'></a>

## 7. Parte 3 — Integração ML + Fuzzy (Abordagem B)

Pipeline encadeado:

1. O melhor modelo de ML recebe os 6 atributos do estudante e devolve `predict_proba`
2. A probabilidade da classe *alto* (índice 2) é extraída como `prob_evasao_ml`
3. Essa probabilidade + `frequencia` + `acessos_ava` alimentam o sistema Fuzzy
4. O Fuzzy devolve um **score 0–100** e um **rótulo linguístico**

**Por que não usar só o ML?** O ML classifica com alta acurácia, mas o output
`alto` ou `0.73` é opaco para um coordenador pedagógico. O Fuzzy transforma
isso em `crítico, 88/100` — com urgência imediata e acionável. Além disso,
o Fuzzy incorpora frequência e engajamento com semântica contextual,
permitindo nuances que a classificação discreta não expressa.

In [ ]:
# ─── Pipeline ML → Fuzzy ──────────────────────────────────────────────────────

probabilidades_ml = melhor_modelo.predict_proba(X_teste_melhor)
predicoes_ml      = melhor_modelo.predict(X_teste_melhor)

# Extrai P(evasão = alto) — índice 2 no mapeamento 0=baixo, 1=medio, 2=alto
prob_classe_alto = probabilidades_ml[:, 2]

X_teste_idx  = X_teste.reset_index(drop=True)
y_teste_idx  = y_teste.reset_index(drop=True)

print(f'Aplicando pipeline ML → Fuzzy em {len(X_teste_idx)} estudantes...')

registros = []
for i in range(len(X_teste_idx)):
    resultado = executar_inferencia(
        probabilidade_ml = float(prob_classe_alto[i]),
        frequencia       = float(X_teste_idx.iloc[i]['frequencia']),
        engajamento      = float(X_teste_idx.iloc[i]['acessos_ava']),
    )
    registros.append({
        'frequencia'      : X_teste_idx.iloc[i]['frequencia'],
        'nota_media'      : X_teste_idx.iloc[i]['nota_media'],
        'acessos_ava'     : X_teste_idx.iloc[i]['acessos_ava'],
        'prob_ml_alto'    : round(float(prob_classe_alto[i]), 4),
        'predicao_ml'     : MAPA_INVERSO[predicoes_ml[i]],
        'real'            : MAPA_INVERSO[y_teste_idx.iloc[i]],
        'score_fuzzy'     : resultado['score'],
        'rotulo_fuzzy'    : resultado['rotulo'],
    })

df_resultado = pd.DataFrame(registros)
df_resultado.to_csv('resultados_integracao.csv', index=False)

print('Concluído. Salvo em resultados_integracao.csv')
df_resultado.head()

In [ ]:
# ─── Análise de coerência ML ↔ Fuzzy ─────────────────────────────────────────
# 'critico' do Fuzzy equivale a 'alto' para comparação com os rótulos do ML.

MAPA_ROTULO_FUZZY = {'baixo': 'baixo', 'medio': 'medio', 'alto': 'alto', 'critico': 'alto'}
df_resultado['rotulo_fuzzy_norm'] = df_resultado['rotulo_fuzzy'].map(MAPA_ROTULO_FUZZY)

concordancia = (df_resultado['predicao_ml'] == df_resultado['rotulo_fuzzy_norm']).mean()
acc_ml_vs_real    = (df_resultado['predicao_ml']       == df_resultado['real']).mean()
acc_fuzzy_vs_real = (df_resultado['rotulo_fuzzy_norm'] == df_resultado['real']).mean()

print(f'Concordância ML ↔ Fuzzy  : {concordancia:.1%}')
print(f'Acurácia ML vs Real      : {acc_ml_vs_real:.1%}')
print(f'Acurácia Fuzzy vs Real   : {acc_fuzzy_vs_real:.1%}')
print()
print('Tabela cruzada ML × Fuzzy (% por linha):')
print((pd.crosstab(df_resultado['predicao_ml'],
                   df_resultado['rotulo_fuzzy_norm'],
                   rownames=['ML'], colnames=['Fuzzy'],
                   normalize='index') * 100).round(1).to_string())

In [ ]:
# ─── Visualizações da integração ──────────────────────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(13, 10))

# ─── Score Fuzzy por classe real ──────────────────────────────────────────────
ax = axes[0, 0]
sns.boxplot(data=df_resultado, x='real', y='score_fuzzy',
            order=ORDEM_CLASSES, palette=PALETA_RISCO, ax=ax)
for limiar, cor in [(30, 'green'), (55, 'orange'), (75, 'red')]:
    ax.axhline(limiar, color=cor, ls='--', lw=1.4, alpha=0.65)
ax.set_title('Score Fuzzy por Classe Real', fontweight='bold')
ax.set_xlabel('Classe Real')
ax.set_ylabel('Score (0–100)')

# ─── Probabilidade ML vs Score Fuzzy ─────────────────────────────────────────
ax = axes[0, 1]
for classe, cor in PALETA_RISCO.items():
    subset = df_resultado[df_resultado['real'] == classe]
    ax.scatter(subset['prob_ml_alto'], subset['score_fuzzy'],
               c=cor, alpha=0.5, s=18, label=classe.capitalize())
# Linha de tendência linear
z  = np.polyfit(df_resultado['prob_ml_alto'], df_resultado['score_fuzzy'], 1)
xs = np.linspace(0, 1, 100)
ax.plot(xs, np.poly1d(z)(xs), 'k--', lw=1.4, alpha=0.65, label='Tendência')
ax.set_xlabel('P(evasão = alto) — ML')
ax.set_ylabel('Score Fuzzy')
ax.set_title('Probabilidade ML vs Score Fuzzy', fontweight='bold')
ax.legend(fontsize=8)

# ─── Distribuição dos rótulos fuzzy ──────────────────────────────────────────
ax = axes[1, 0]
contagem_rotulos = df_resultado['rotulo_fuzzy'].value_counts()
PALETA_FUZZY     = {'baixo': '#55A868', 'medio': '#4C72B0',
                    'alto': '#DD8452',  'critico': '#C44E52'}
ax.pie(contagem_rotulos.values,
       labels=[f'{r.capitalize()} (n={v})'
               for r, v in zip(contagem_rotulos.index, contagem_rotulos.values)],
       colors=[PALETA_FUZZY.get(r, '#aaa') for r in contagem_rotulos.index],
       autopct='%1.1f%%', startangle=90,
       wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
ax.set_title('Rótulos do Sistema Fuzzy — Conjunto de Teste', fontweight='bold')

# ─── Histograma de scores por classe real ────────────────────────────────────
ax = axes[1, 1]
for classe, cor in PALETA_RISCO.items():
    subset = df_resultado[df_resultado['real'] == classe]
    ax.hist(subset['score_fuzzy'], bins=18, alpha=0.55, color=cor,
            label=f'{classe.capitalize()} (n={len(subset)})')
for limiar, cor in [(30, 'green'), (55, 'orange'), (75, 'red')]:
    ax.axvline(limiar, color=cor, ls='--', lw=1.4, alpha=0.65)
ax.set_xlabel('Score Fuzzy (0–100)')
ax.set_title('Distribuição dos Scores por Classe Real', fontweight='bold')
ax.legend(fontsize=9)

fig.suptitle('Integração ML + Fuzzy — Análise de Coerência',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
<a id='resultados'></a>

## 8. Resultados e Discussão

In [ ]:
# ─── Resumo dos resultados ────────────────────────────────────────────────────

scores_alto  = df_resultado[df_resultado['real'] == 'alto']['score_fuzzy']
scores_baixo = df_resultado[df_resultado['real'] == 'baixo']['score_fuzzy']

print('MACHINE LEARNING')
print(f'  Árvore de Decisão   acc={arvore_acc:.3f}  '
      f'F1={arvore_rep["macro avg"]["f1-score"]:.3f}  '
      f'CV={arvore_cv.mean():.3f}±{arvore_cv.std():.3f}')
print(f'  Naive Bayes         acc={nb_acc:.3f}  '
      f'F1={nb_rep["macro avg"]["f1-score"]:.3f}  '
      f'CV={nb_cv.mean():.3f}±{nb_cv.std():.3f}')
print(f'  Selecionado         {melhor_nome}')
print()
print('SISTEMA FUZZY')
print(f'  Entradas    : prob_evasao_ml · frequencia · engajamento_ava')
print(f'  Regras      : 8 (Mamdani)')
print(f'  Score alto  : {scores_alto.mean():.1f} ± {scores_alto.std():.1f}')
print(f'  Score baixo : {scores_baixo.mean():.1f} ± {scores_baixo.std():.1f}')
print(f'  Separação   : {scores_alto.mean() - scores_baixo.mean():.1f} pontos')
print()
print('INTEGRAÇÃO')
print(f'  Concordância ML ↔ Fuzzy : {concordancia:.1%}')
print(f'  Acurácia ML   vs Real   : {acc_ml_vs_real:.1%}')
print(f'  Acurácia Fuzzy vs Real  : {acc_fuzzy_vs_real:.1%}')

In [ ]:
# ─── Discussão crítica ────────────────────────────────────────────────────────

print("""
DISCUSSÃO CRÍTICA
─────────────────────────────────────────────────────────────────────

1. DESEMPENHO DOS MODELOS DE ML
   · Naive Bayes (~92%) superou a Árvore (~84%).
   · O baixo desvio padrão no CV (±0.02 para NB, ±0.04 para Árvore)
     indica boa generalização — nenhum dos dois modelos está overfittando.
   · A Árvore, mesmo com acurácia inferior, tem valor complementar:
     suas regras IF-THEN são auditáveis e dialoga com as regras fuzzy.

2. COERÊNCIA DO SISTEMA FUZZY
   · Separação de ~67 pontos entre alto risco (≈84) e baixo risco (≈17)
     valida que as funções de pertinência e as regras estão calibradas.
   · O quarto termo "crítico" (score > 75) distingue casos de emergência
     real de casos apenas com risco elevado — granularidade que o ML não fornece.

3. O QUE A INTEGRAÇÃO AGREGA
   · Concordância de ~86% demonstra que os sistemas são complementares.
   · Os ~14% divergentes são informativos: quando o ML classifica como "alto"
     mas o Fuzzy retorna "médio", pode indicar que a frequência atual está
     estável — o Fuzzy incorpora contexto que o ML não viu no treino.
   · "crítico, 88/100" comunica urgência diferente de "alto, 66/100",
     distinção que a classificação discreta do ML não é capaz de fazer.

4. LIMITAÇÕES
   · Base sintética — distribuições controladas podem não capturar
     toda a complexidade de dados reais (sazonalidade, eventos externos).
   · Funções de pertinência definidas por heurística pedagógica;
     em produção, seriam calibradas com especialistas da área.
   · O sistema Fuzzy usa apenas 3 das 6 variáveis disponíveis —
     incluir nota_media e situacao_financeira enriqueceria a saída.

5. MELHORIAS FUTURAS
   · Usar base real de IES parceira (LGPD-compliant).
   · ANFIS (Adaptive Neuro-Fuzzy Inference System) para calibrar
     automaticamente as funções de pertinência com os dados.
   · Adicionar variáveis temporais: tendência de queda de frequência
     nos últimos N períodos seria um preditor mais forte.
   · Dashboard interativo (Streamlit) para uso pelos coordenadores.
""")

---
<a id='conclusao'></a>

## 9. Conclusão

Desenvolvemos um sistema inteligente integrado para estimação de risco de
evasão acadêmica utilizando **Abordagem B** — pipeline ML → Fuzzy.

**Síntese:**
base sintética balanceada (420 registros, 6 features) · EDA completa ·
dois modelos de ML com validação cruzada e métricas completas ·
sistema Fuzzy Mamdani (3 entradas, 8 regras, 4 termos de saída,
defuzzificação por centroide) · pipeline integrado · análise de coerência.

**Conceitos da disciplina aplicados:**
aprendizado supervisionado · classificação multiclasse · Teorema de Bayes ·
impureza de Gini · validação cruzada · conjuntos fuzzy (Zadeh, 1965) ·
variáveis linguísticas · fuzzificação · inferência Mamdani ·
defuzzificação por centroide.

---

## Referências

RUSSELL, Stuart; NORVIG, Peter. **Inteligência Artificial**. 3. ed. Elsevier, 2013.

ZADEH, Lotfi A. Fuzzy Sets. **Information and Control**, v. 8, n. 3, p. 338–353, 1965.

SCIKIT-LEARN. *User Guide*. https://scikit-learn.org/stable/user_guide.html. Acesso: jun. 2025.

SCIKIT-FUZZY. *Documentation*. https://pythonhosted.org/scikit-fuzzy/. Acesso: jun. 2025.

MALVEZZI, William. **Slides de IA: Lógica Fuzzy e Machine Learning**. Material interno, 2025.

In [ ]:
# ─── Checklist de entregáveis — AVA (25/06/2025) ─────────────────────────────

import os

arquivos_gerados = [
    'base_evasao_academica.csv',
    'resultados_integracao.csv',
]
for arquivo in arquivos_gerados:
    status = '✅' if os.path.exists(arquivo) else '⬜ (gerado ao executar)'
    print(f'{status}  {arquivo}')

print()
print('Entregar no AVA:')
print('  ✅  risco_evasao.ipynb')
print('  ✅  base_evasao_academica.csv')
print('  ✅  README.md')
print('  ⬜  relatório final (PDF — 10 a 20 páginas)')
print('  ⬜  apresentação (PPTX ou PDF)')